In [5]:
import ssl
import certifi
import urllib.request

# Patch SSL to use certifi's certificates instead of Windows store
ssl._create_default_https_context = ssl.create_default_context
ssl._create_default_https_context = lambda: ssl.create_default_context(cafile=certifi.where())

import ee
import geemap
import geopandas as gpd

# ee.Authenticate()
ee.Initialize(project='oceanic-gecko-495018-r2')


In [6]:
# STEP 2: Load the GeoJSON from your local repo folder
# Make sure the file path matches where you saved it in your repo
print("Loading local GeoJSON...")
gdf = gpd.read_file('../../data/INDIA_DISTRICTS.geojson')
print(gdf.columns)


Loading local GeoJSON...
Index(['objectid', 'statecode', 'district', 'state', 'shape_leng',
       'shape_area', 'dist_code', 'D_CODE', 'st_code', 'remarks', 'geometry'],
      dtype='str')


In [7]:
# STEP 3: Filter for the toxic gas chamber states (not MP)
# We are slicing out Punjab and Haryana to lock in our agricultural belt.
agri_belt_gdf = gdf[gdf['state'].isin(['PUNJAB', 'HARYANA'])]

# STEP 4: Convert local geometry to Earth Engine geometry
# This translates your local file into a format Google's servers understand
ee_agri_belt = geemap.geopandas_to_ee(agri_belt_gdf)

print("Target regions locked in. Ready to pull the satellite data.")

Target regions locked in. Ready to pull the satellite data.


In [8]:
# STEP 5: Pulling the Sentinel-2 Satellite Data
# We target the peak stubble burning season (Sept - Nov)
start_date = '2025-09-01'
end_date = '2025-11-30'

# The Cloud Masking function (because clouds = trash data)
def mask_s2_clouds(image):
    qa = image.select('QA60')
    # Bits 10 and 11 are clouds and cirrus. We want them to be 0 (clear).
    cloudBitMask = 1 << 10
    cirrusBitMask = 1 << 11
    mask = qa.bitwiseAnd(cloudBitMask).eq(0).And(qa.bitwiseAnd(cirrusBitMask).eq(0))
    return image.updateMask(mask).divide(10000)

print("Fetching Sentinel-2 data... this might take a sec bro.")

# Pulling the imagery specifically for your Punjab/Haryana polygon
dataset = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED') \
    .filterBounds(ee_agri_belt) \
    .filterDate(start_date, end_date) \
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20)) \
    .map(mask_s2_clouds)

# We take the median of all images in that date range to get one clean, cloud-free map
median_image = dataset.median()

# The Magic Math: NDVI = (NIR - Red) / (NIR + Red)
# For Sentinel-2: NIR is Band 8 (B8), Red is Band 4 (B4)
ndvi = median_image.normalizedDifference(['B8', 'B4']).rename('NDVI')

# STEP 6: Visualizing the gas chamber on a map
print("Rendering map...")
Map = geemap.Map()
Map.centerObject(ee_agri_belt, 6) # Zoom level 6 is perfect for a regional view

# Adding the NDVI layer with a dope color palette
# Red = dead/harvested/bare soil, Green = healthy growing crops
ndvi_params = {
    'min': 0.0,
    'max': 1.0,
    'palette': ['red', 'yellow', 'green'] 
}

# Clip the satellite data so it perfectly fits inside the state borders, no spilling over
Map.addLayer(ndvi.clip(ee_agri_belt), ndvi_params, 'NDVI (Plant Health)')

# Add an outline of the states just to make it look professional
Map.addLayer(ee.Image().paint(ee_agri_belt, 0, 2), {'palette': ['black']}, 'Punjab & Haryana Borders')

print("Map generated. Scroll down to look at it.")
Map

Fetching Sentinel-2 data... this might take a sec bro.
Rendering map...
Map generated. Scroll down to look at it.


Map(center=[30.071654782562103, 75.85202480540147], controls=(WidgetControl(options=['position', 'transparent_…

## What does this map represent:

* yellow -> orange -> red: this means the crop is being harvested, like low chlorophyll is observed
* green: this means the crop is standing / there is a forest 

#### what it means for us:
when a farm's patch transitions from *green* to *orange*, the moment it turns *red*, we need to dispatch the trucks in less than 20 days so that the truck reaches the farms before farmer burns the stubble

In [9]:
# STEP 7: Actually saving the data so you don't lose your mind
import os

print("Starting the download... do not close VS Code.")

# We save it as a GeoTIFF file right in your current GitHub folder
out_file = '../../data/punjab_haryana_ndvi.tif'

# Warning: DO NOT change the scale to 10 yet. 
# Sentinel-2 is 10-meter resolution. If you try to download 10m pixels 
# for TWO ENTIRE STATES at once, your laptop will literally melt.
# We use scale=1000 (1km resolution) first just to make sure the pipeline works.

geemap.ee_export_image(
    ndvi.clip(ee_agri_belt), 
    filename=out_file, 
    scale=1000, 
    region=ee_agri_belt.geometry(),
    file_per_band=False
)

print(f"Boom. Saved locally as {out_file}. You can sleep now.")

Starting the download... do not close VS Code.
Generating URL ...
Please wait ...
Data downloaded to d:\Coding\Hackathon\samsung\agri-Waste-biomass-valuation-engine\data\punjab_haryana_ndvi.tif
Boom. Saved locally as ../../data/punjab_haryana_ndvi.tif. You can sleep now.


In [10]:
# STEP 8: The "Big Brain" Cropland Mask
print("Summoning the ESA WorldCover dataset...")

# Pull the 2021 ESA WorldCover map
world_cover = ee.ImageCollection("ESA/WorldCover/v200").first()

# In their system, Class 40 = Cropland. 
# We create a binary mask: 1 if it's a farm, 0 if it's anything else.
crop_mask = world_cover.eq(40).clip(ee_agri_belt)

# Now we apply this mask to your existing NDVI map
# This literally erases all non-farm pixels from existence
farm_only_ndvi = ndvi.updateMask(crop_mask)

# Let's visualize the filtered map
Map2 = geemap.Map()
Map2.centerObject(ee_agri_belt, 6)

# The mask itself (showing you where the farms are)
Map2.addLayer(crop_mask.selfMask(), {'palette': ['blue']}, 'Identified Farmlands')

# The NDVI data, but ONLY inside the farms
Map2.addLayer(farm_only_ndvi, ndvi_params, 'NDVI (Farms Only)')

print("Boom. Cities and roads deleted. We only track farms now.")
Map2

Summoning the ESA WorldCover dataset...
Boom. Cities and roads deleted. We only track farms now.


Map(center=[30.071654782562103, 75.85202480540147], controls=(WidgetControl(options=['position', 'transparent_…

earlier it was pulling out data for all parts, now it only does for the agricultural lands in the are (highways, cities... removed)